# Notebook 27 — Where Universality Breaks

This notebook extends the effective-noise universality analysis by asking:

**When does the shared effective-noise geometry stop being a good description?**

Earlier notebooks suggested that nearby phase-locked protocols often share:
- a similar effective direction
  `gamma_eff = gamma + lambda * gamma_phi`,
- similar 1D response curves,
- similar rapid-degradation bands.

Here we deliberately sweep coherent-control settings away from the baseline and measure where that universality begins to fail.

We track:
1. best-fit `lambda`,
2. axis-slice collapse loss,
3. response-curve mismatch from the baseline,
4. approximate universality-break indicators in control space.

This notebook is meant to identify the boundary between:
- a locally universal phase-locked family,
- and protocol regions where the reduced noise geometry changes qualitatively.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'matplotlib', 'scipy']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.optimize import minimize_scalar
from qutip import basis, qeye, tensor, sigmax, mesolve


## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
n = r * r.dag()
sigma_minus = g * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = [gg, gr, rg, rr]

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
n1 = tensor(n, I)
n2 = tensor(I, n)
sm1 = tensor(sigma_minus, I)
sm2 = tensor(I, sigma_minus)

U_cz = np.diag([1, 1, 1, -1]).astype(complex)


## Baseline protocol and broader control sweeps

In [ ]:
baseline = {
    'T': 26.0,
    'alpha': 0.10,
    'omega_max': 1.09,
    'delta0': 1.00,
    'p': 1.00,
    'q': 0.72,
}
V = 4.0

T_values = [18.0, 22.0, 26.0, 30.0, 34.0]
alpha_values = [0.04, 0.08, 0.10, 0.12, 0.16]
omega_values = [0.90, 1.00, 1.09, 1.18, 1.30]

print("Baseline:", baseline)
print("T sweep:", T_values)
print("alpha sweep:", alpha_values)
print("omega sweep:", omega_values)


## Shaped schedules and Hamiltonian

In [ ]:
def omega_shaped(t, T, omega_max, p):
    s = t / T
    base = 16.0 * (s * (1.0 - s)) ** 2
    return omega_max * np.maximum(base, 0.0) ** p

def delta_shaped(t, T, V, alpha, delta0, q):
    s = t / T
    shaped = s ** q
    return delta0 + alpha * V * 0.5 * (1.0 - np.cos(np.pi * shaped))

H_omega = 0.5 * (sx1 + sx2)
H_delta = -(n1 + n2)
H_V = n1 * n2

def build_time_dependent_hamiltonian(T, omega_max, alpha, V, delta0, p, q):
    def coeff_omega(t, args=None):
        return omega_shaped(t, T=T, omega_max=omega_max, p=p)
    def coeff_delta(t, args=None):
        return delta_shaped(t, T=T, V=V, alpha=alpha, delta0=delta0, q=q)
    return [
        [H_omega, coeff_omega],
        [H_delta, coeff_delta],
        [H_V, lambda t, args=None: V],
    ]


## Collapse operators

In [ ]:
def collapse_ops(gamma_decay=0.0, gamma_phi=0.0):
    ops = []
    if gamma_decay > 0:
        ops.append(np.sqrt(gamma_decay) * sm1)
        ops.append(np.sqrt(gamma_decay) * sm2)
    if gamma_phi > 0:
        ops.append(np.sqrt(gamma_phi) * n1)
        ops.append(np.sqrt(gamma_phi) * n2)
    return ops


## Noisy evolution and effective map

In [ ]:
def run_noisy_evolution(psi0, params, gamma_decay=0.0, gamma_phi=0.0, n_steps=160):
    H = build_time_dependent_hamiltonian(
        params['T'], params['omega_max'], params['alpha'], V, params['delta0'], params['p'], params['q']
    )
    times = np.linspace(0.0, params['T'], n_steps)
    c_ops = collapse_ops(gamma_decay=gamma_decay, gamma_phi=gamma_phi)
    result = mesolve(H, psi0, times, c_ops)
    return result.states[-1]

def state_to_column_mixed(state):
    vals = []
    for basis_state in basis_states:
        if state.isket:
            amp = basis_state.overlap(state)
            vals.append(amp)
        else:
            val = (basis_state.dag() * state * basis_state)
            vals.append(np.real(val.full()[0, 0]) if hasattr(val, 'full') else np.real(val))
    return np.array(vals, dtype=complex)

def build_noisy_effective_map(params, gamma_decay=0.0, gamma_phi=0.0, n_steps=160):
    cols = []
    finals = []
    for psi0 in basis_states:
        final_state = run_noisy_evolution(
            psi0, params, gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=n_steps
        )
        cols.append(state_to_column_mixed(final_state))
        finals.append(final_state)
    return np.column_stack(cols), finals


## Diagnostics

In [ ]:
def process_fidelity(U_eff, U_target):
    d = U_target.shape[0]
    return float(np.abs(np.trace(np.conjugate(U_target.T) @ U_eff)) ** 2 / (d ** 2))

def wrapped_phase(x):
    return np.angle(np.exp(1j * x))

def diagonal_phases(U_eff):
    return np.angle(np.diag(U_eff))

def entangling_phase(U_eff):
    phi = diagonal_phases(U_eff)
    return wrapped_phase(phi[0] + phi[3] - phi[1] - phi[2])

def extract_local_z_phases(U_eff):
    phi00, phi01, phi10, phi11 = diagonal_phases(U_eff)
    phi_ent = entangling_phase(U_eff)
    a = wrapped_phase(phi10 - phi00)
    b = wrapped_phase(phi01 - phi00)
    global_phase = phi00
    return global_phase, a, b, phi_ent

def local_z_diagonal(a, b):
    return np.diag([1.0, np.exp(1j*b), np.exp(1j*a), np.exp(1j*(a+b))]).astype(complex)

def compensate_local_z(U_eff):
    global_phase, a, b, phi_ent = extract_local_z_phases(U_eff)
    U1 = np.exp(-1j * global_phase) * U_eff
    U2 = U1 @ local_z_diagonal(-a, -b)
    return U2, global_phase, a, b, phi_ent

def compensated_cz_fidelity(U_eff):
    U_comp, _, _, _, _ = compensate_local_z(U_eff)
    return process_fidelity(U_comp, U_cz)

def leakage_from_finals(final_states):
    scores = []
    for idx, final_state in enumerate(final_states):
        basis_state = basis_states[idx]
        if final_state.isket:
            amp = basis_state.overlap(final_state)
            score = np.abs(amp) ** 2
        else:
            val = (basis_state.dag() * final_state * basis_state)
            score = np.real(val.full()[0, 0]) if hasattr(val, 'full') else np.real(val)
        scores.append(score)
    return float(1.0 - np.mean(scores))

def coherence_proxy(final_states):
    vals = []
    for state in final_states:
        rho = state.proj() if state.isket else state
        arr = rho.full()
        off = arr.copy()
        np.fill_diagonal(off, 0.0)
        vals.append(np.mean(np.abs(off)))
    return float(np.mean(vals))


## Shared noise grid

In [ ]:
gamma_decay_vals = np.linspace(0.0, 0.12, 17)
gamma_phi_vals = np.linspace(0.0, 0.12, 17)


## Helper: evaluate one protocol variant

In [ ]:
def evaluate_protocol(params):
    cz_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
    coh_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
    leak_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))

    for i, gamma_phi in enumerate(gamma_phi_vals):
        for j, gamma_decay in enumerate(gamma_decay_vals):
            U_eff, finals = build_noisy_effective_map(
                params, gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=160
            )
            cz_map[i, j] = compensated_cz_fidelity(U_eff)
            coh_map[i, j] = coherence_proxy(finals)
            leak_map[i, j] = leakage_from_finals(finals)

    S = cz_map * coh_map * (1.0 - leak_map)
    S_norm = S / np.max(S) if np.max(S) > 0 else S.copy()

    S_gamma = S_norm[0, :]
    S_phi = S_norm[:, 0]
    interp_gamma = interp1d(gamma_decay_vals, S_gamma, kind='linear', fill_value='extrapolate')

    def loss(lam):
        gamma_eff_phi = lam * gamma_phi_vals
        pred = interp_gamma(np.clip(gamma_eff_phi, gamma_decay_vals.min(), gamma_decay_vals.max()))
        return float(np.mean((pred - S_phi) ** 2))

    fit = minimize_scalar(loss, bounds=(0.1, 5.0), method='bounded')
    lam = float(fit.x)
    axis_loss = float(fit.fun)

    gamma_eff_grid = np.zeros_like(S_norm)
    for i, gamma_phi in enumerate(gamma_phi_vals):
        for j, gamma_decay in enumerate(gamma_decay_vals):
            gamma_eff_grid[i, j] = gamma_decay + lam * gamma_phi

    gamma_eff_flat = gamma_eff_grid.ravel()
    S_flat = S_norm.ravel()

    order = np.argsort(gamma_eff_flat)
    gamma_eff_sorted = gamma_eff_flat[order]
    S_sorted = S_flat[order]

    n_bins = 16
    bins = np.linspace(gamma_eff_sorted.min(), gamma_eff_sorted.max(), n_bins + 1)
    centers = 0.5 * (bins[:-1] + bins[1:])
    means = np.full(n_bins, np.nan)
    for k in range(n_bins):
        mask = (gamma_eff_sorted >= bins[k]) & (gamma_eff_sorted < bins[k+1])
        if np.any(mask):
            means[k] = np.mean(S_sorted[mask])

    valid = ~np.isnan(means)
    centers = centers[valid]
    means = means[valid]

    dy = np.gradient(means, centers)
    d2y = np.gradient(dy, centers)
    peak_drop_x = float(centers[int(np.argmin(dy))])

    return {
        'params': params,
        'S_norm': S_norm,
        'lambda': lam,
        'axis_loss': axis_loss,
        'curve_x': centers,
        'curve_y': means,
        'peak_drop_x': peak_drop_x,
    }


## Evaluate sweeps

In [ ]:
results_T = {}
for T in T_values:
    params = {**baseline, 'T': T}
    results_T[T] = evaluate_protocol(params)
    print("done T =", T)

results_alpha = {}
for a in alpha_values:
    params = {**baseline, 'alpha': a}
    results_alpha[a] = evaluate_protocol(params)
    print("done alpha =", a)

results_omega = {}
for om in omega_values:
    params = {**baseline, 'omega_max': om}
    results_omega[om] = evaluate_protocol(params)
    print("done omega_max =", om)


## Baseline reference

In [ ]:
baseline_res = results_T[baseline['T']]
lambda0 = baseline_res['lambda']
curve0_x = baseline_res['curve_x']
curve0_y = baseline_res['curve_y']

interp0 = interp1d(curve0_x, curve0_y, kind='linear', fill_value='extrapolate')
print("baseline lambda =", lambda0)


## Universality-break metrics

In [ ]:
def curve_mismatch_to_baseline(x, y):
    y_ref = interp0(np.clip(x, curve0_x.min(), curve0_x.max()))
    return float(np.mean((y - y_ref) ** 2))

for sweep in (results_T, results_alpha, results_omega):
    for key, res in sweep.items():
        res['lambda_shift'] = abs(res['lambda'] - lambda0)
        res['curve_mismatch'] = curve_mismatch_to_baseline(res['curve_x'], res['curve_y'])
        res['peak_shift'] = abs(res['peak_drop_x'] - baseline_res['peak_drop_x'])


## T sweep diagnostics

In [ ]:
T_keys = list(results_T.keys())
lam_T = [results_T[k]['lambda'] for k in T_keys]
loss_T = [results_T[k]['axis_loss'] for k in T_keys]
shift_T = [results_T[k]['lambda_shift'] for k in T_keys]
curve_T = [results_T[k]['curve_mismatch'] for k in T_keys]
peak_T = [results_T[k]['peak_shift'] for k in T_keys]

fig, axes = plt.subplots(2, 2, figsize=(11, 8))

axes[0,0].plot(T_keys, lam_T, marker='o')
axes[0,0].axhline(lambda0, linestyle='--')
axes[0,0].set_title('T sweep: lambda')
axes[0,0].set_xlabel('T')
axes[0,0].set_ylabel('best-fit lambda')

axes[0,1].plot(T_keys, loss_T, marker='o')
axes[0,1].set_title('T sweep: axis loss')
axes[0,1].set_xlabel('T')
axes[0,1].set_ylabel('axis-slice MSE')

axes[1,0].plot(T_keys, curve_T, marker='o')
axes[1,0].set_title('T sweep: response-curve mismatch')
axes[1,0].set_xlabel('T')
axes[1,0].set_ylabel('curve mismatch')

axes[1,1].plot(T_keys, peak_T, marker='o')
axes[1,1].set_title('T sweep: rapid-drop shift')
axes[1,1].set_xlabel('T')
axes[1,1].set_ylabel('peak-drop shift')

plt.tight_layout()
plt.show()


## Alpha sweep diagnostics

In [ ]:
A_keys = list(results_alpha.keys())
lam_A = [results_alpha[k]['lambda'] for k in A_keys]
loss_A = [results_alpha[k]['axis_loss'] for k in A_keys]
curve_A = [results_alpha[k]['curve_mismatch'] for k in A_keys]
peak_A = [results_alpha[k]['peak_shift'] for k in A_keys]

fig, axes = plt.subplots(2, 2, figsize=(11, 8))

axes[0,0].plot(A_keys, lam_A, marker='o')
axes[0,0].axhline(lambda0, linestyle='--')
axes[0,0].set_title('alpha sweep: lambda')
axes[0,0].set_xlabel('alpha')
axes[0,0].set_ylabel('best-fit lambda')

axes[0,1].plot(A_keys, loss_A, marker='o')
axes[0,1].set_title('alpha sweep: axis loss')
axes[0,1].set_xlabel('alpha')
axes[0,1].set_ylabel('axis-slice MSE')

axes[1,0].plot(A_keys, curve_A, marker='o')
axes[1,0].set_title('alpha sweep: response-curve mismatch')
axes[1,0].set_xlabel('alpha')
axes[1,0].set_ylabel('curve mismatch')

axes[1,1].plot(A_keys, peak_A, marker='o')
axes[1,1].set_title('alpha sweep: rapid-drop shift')
axes[1,1].set_xlabel('alpha')
axes[1,1].set_ylabel('peak-drop shift')

plt.tight_layout()
plt.show()


## Omega sweep diagnostics

In [ ]:
O_keys = list(results_omega.keys())
lam_O = [results_omega[k]['lambda'] for k in O_keys]
loss_O = [results_omega[k]['axis_loss'] for k in O_keys]
curve_O = [results_omega[k]['curve_mismatch'] for k in O_keys]
peak_O = [results_omega[k]['peak_shift'] for k in O_keys]

fig, axes = plt.subplots(2, 2, figsize=(11, 8))

axes[0,0].plot(O_keys, lam_O, marker='o')
axes[0,0].axhline(lambda0, linestyle='--')
axes[0,0].set_title('omega_max sweep: lambda')
axes[0,0].set_xlabel('omega_max')
axes[0,0].set_ylabel('best-fit lambda')

axes[0,1].plot(O_keys, loss_O, marker='o')
axes[0,1].set_title('omega_max sweep: axis loss')
axes[0,1].set_xlabel('omega_max')
axes[0,1].set_ylabel('axis-slice MSE')

axes[1,0].plot(O_keys, curve_O, marker='o')
axes[1,0].set_title('omega_max sweep: response-curve mismatch')
axes[1,0].set_xlabel('omega_max')
axes[1,0].set_ylabel('curve mismatch')

axes[1,1].plot(O_keys, peak_O, marker='o')
axes[1,1].set_title('omega_max sweep: rapid-drop shift')
axes[1,1].set_xlabel('omega_max')
axes[1,1].set_ylabel('peak-drop shift')

plt.tight_layout()
plt.show()


## Compare response curves directly

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))

for T in T_keys:
    axes[0].plot(results_T[T]['curve_x'], results_T[T]['curve_y'], label=f'T={T:g}')
axes[0].set_title('T sweep response curves')
axes[0].set_xlabel('gamma_eff')
axes[0].set_ylabel('Normalized S')
axes[0].legend(fontsize=8)

for a in A_keys:
    axes[1].plot(results_alpha[a]['curve_x'], results_alpha[a]['curve_y'], label=f'α={a:g}')
axes[1].set_title('alpha sweep response curves')
axes[1].set_xlabel('gamma_eff')
axes[1].set_ylabel('Normalized S')
axes[1].legend(fontsize=8)

for om in O_keys:
    axes[2].plot(results_omega[om]['curve_x'], results_omega[om]['curve_y'], label=f'Ω={om:g}')
axes[2].set_title('omega sweep response curves')
axes[2].set_xlabel('gamma_eff')
axes[2].set_ylabel('Normalized S')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()


## Representative universality-break examples

In [ ]:
show_T = [18.0, 26.0, 34.0]
fig, axes = plt.subplots(1, len(show_T), figsize=(12, 4.2))

for ax, T in zip(axes, show_T):
    arr = results_T[T]['S_norm']
    ax.imshow(
        arr,
        origin='lower',
        aspect='auto',
        extent=[gamma_decay_vals[0], gamma_decay_vals[-1], gamma_phi_vals[0], gamma_phi_vals[-1]],
        vmin=0, vmax=1
    )
    ax.set_title(f'T={T:g}')
    ax.set_xlabel('gamma')
    ax.set_ylabel('gamma_phi')

plt.tight_layout()
plt.show()


## Compact summary

In [ ]:
def summarize_extremes(results, label):
    keys = list(results.keys())
    lambda_shift = np.array([results[k]['lambda_shift'] for k in keys])
    curve_mismatch = np.array([results[k]['curve_mismatch'] for k in keys])
    peak_shift = np.array([results[k]['peak_shift'] for k in keys])

    k_lambda = keys[int(np.argmax(lambda_shift))]
    k_curve = keys[int(np.argmax(curve_mismatch))]
    k_peak = keys[int(np.argmax(peak_shift))]

    lines = []
    lines.append(f"{label} sweep:")
    lines.append(f"- largest lambda shift at {k_lambda}: {results[k_lambda]['lambda_shift']:.4f}")
    lines.append(f"- largest curve mismatch at {k_curve}: {results[k_curve]['curve_mismatch']:.6e}")
    lines.append(f"- largest peak-drop shift at {k_peak}: {results[k_peak]['peak_shift']:.4f}")
    return lines

lines = []
lines.append("Where universality breaks — summary")
lines.append("")
lines.append(f"Baseline lambda = {lambda0:.4f}")
lines.append("")
lines.extend(summarize_extremes(results_T, "T"))
lines.append("")
lines.extend(summarize_extremes(results_alpha, "alpha"))
lines.append("")
lines.extend(summarize_extremes(results_omega, "omega_max"))
lines.append("")
lines.append("Interpretation:")
lines.append("- Small lambda shift + small curve mismatch => universality holds locally.")
lines.append("- Large lambda shift or curve mismatch => effective-noise geometry is changing.")
lines.append("- This identifies control-space directions where the one-parameter reduction remains trustworthy, and where it breaks down.")

summary_text = "\n".join(lines)
print(summary_text)


## Final conclusion

This notebook identifies where the reduced effective-noise picture remains stable and where it begins to fail.

That gives a more precise interpretation of universality:

- it is not just about fitting one protocol,
- it is about the persistence of a shared noise geometry across nearby control settings,
- and it eventually breaks when coherent-control changes become large enough to distort the fitted direction or the response curve itself.

So the final statement becomes:

**the phase-locked CZ regime has a locally universal effective noise geometry, but that universality breaks along identifiable directions in control space.**
